# REACT 2026 Datathon - Temporal Fraud Detection (v3_v9)

**One measurement nobody has taken, paid for by deleting four things that were measured as
worthless.**

Twelve runs have put `recent` and `w4` between 0.6133 and 0.6197. Recency (four attempts),
top-of-list re-ranking (hurt by 0.002 on every fold), stationary features (neutral, bad for
the network), segment-aware weights (+0.0002), drift pruning (removed zero features, twice),
tree size, learning rate, feature pruning - all dead ends. Adding a thirteenth guess is not
the move. Measuring the one thing no fold can see is.

### The question

Every model here leans on fraud-rate encodings: `te_merchant`, `ted_merchant`, `te_loc`. In
validation, a row reads those encodings computed from labels **up to its own timestamp**.

At scoring time they freeze on 15 July and go **up to 62 days stale**.

No fold in v3, v3_v2, fraud11, v3_v4 or v3_v5 has ever been scored under that condition, so
nobody knows what it costs. The one hint is a reversal across pipelines: when encodings were
frozen per fold (in a separate line of experiments), the boosted trees collapsed and the
network overtook them; when the encodings stayed fresh, the trees won.

### The measurement

For each fold, the same LightGBM is fitted twice on identical rows, differing only in how the
validation block's encodings are built:

* **fresh** - labels up to each row's own timestamp (what every run so far has reported)
* **frozen** - labels cut at the fold's fitting boundary, exactly as a test row will see them

The gap between those two numbers is the staleness cost, in AP, and it is the number that
decides whether the leaderboard will look like the folds.

### The mitigation, carried as a member

`nol` is the same model with every `te_`, `ted_` and `ten_` column removed. It scores lower
when encodings are fresh - they carry real gain - but it cannot lose anything when they go
stale, and its errors differ from the others' for exactly that reason. If the probe shows a
large staleness cost, this member is the reason the run still has a good answer.

### What was cut to fit the hour

| removed | measured result |
|---|---|
| the `w4` fold | scores the same 82,586 rows as `recent`, with an easier embargo |
| the adversarial drift probe | pruned zero features in two consecutive runs |
| the exhaustive weight grid | 795s per run; equal weights landed within 0.0001 every time |
| CatBoost's deep/slow settings | best_iter 882 at depth 8 bought nothing over depth 6 |


In [1]:
# ============================================================================
# 01 | Environment and run configuration
# ============================================================================
import os
import gc
import time
import warnings

import numpy as np
import pandas as pd
from scipy.stats import rankdata
from sklearn.metrics import average_precision_score

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 250)

DATA_DIR = "/kaggle/input/competitions/react-2026-datathon"
if not os.path.exists(os.path.join(DATA_DIR, "train.csv")):
    for alt in ("/kaggle/input/react-2026-datathon", "./data", "."):
        if os.path.exists(os.path.join(alt, "train.csv")):
            DATA_DIR = alt
            break

SEEDS        = [42, 202]
VALID_DAYS   = 21
EMBARGO_DAYS = 1
FAR_EMBARGO  = 30
HORIZON_DAYS = 62
WARMUP_DAYS  = 30
TE_SMOOTH    = 50.0
TE_SMOOTH_HI = 15.0
TE_TAU       = 45.0
EPS          = 1e-6

RULE_MIN_PREC = 0.25
RULE_MIN_REC  = 0.005
RULE_MAX      = 12

MLP_MAX_EPOCHS = 20
MLP_PATIENCE   = 4
MLP_VAL_SEEDS  = [0]
MLP_FIT_SEEDS  = [0, 1]      # the single knob to cut first if the run runs long

HOUR = 3600
DAY  = 86400

_t0 = time.time()
def tick(msg):
    print(f"[{time.time() - _t0:7.1f}s] {msg}")

tick("configuration loaded")


[    0.0s] configuration loaded


In [2]:
# ============================================================================
# 02 | Load the raw stream and order it chronologically
# ============================================================================
train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"), parse_dates=["timestamp"])
test  = pd.read_csv(os.path.join(DATA_DIR, "test.csv"),  parse_dates=["timestamp"])
try:
    sample_sub = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))
except FileNotFoundError:
    sample_sub = pd.DataFrame({"transaction_id": test["transaction_id"], "fraud": 0.0})

print("train:", train.shape, "| test:", test.shape)
print("train window:", train.timestamp.min(), "->", train.timestamp.max())
print("test  window:", test.timestamp.min(),  "->", test.timestamp.max())
print("base fraud rate:", round(train.fraud.mean(), 6))

train["is_train"] = np.int8(1)
test["is_train"]  = np.int8(0)
test["fraud"]     = np.nan

df = pd.concat([train, test], ignore_index=True, sort=False)
df = df.sort_values(["timestamp", "transaction_id"], kind="mergesort").reset_index(drop=True)

TRAIN_END   = train.timestamp.max()
STREAM_START = df["timestamp"].min()
del train, test
gc.collect()

# Epoch seconds computed through a duration rather than a raw int64 cast: pandas resolves
# parsed timestamps to different units across versions, and casting the raw integer would
# silently change the unit of every trailing window below.
df["ts"] = (df["timestamp"] - pd.Timestamp("1970-01-01")).dt.total_seconds().astype("int64")
IS_TRAIN = df["is_train"].to_numpy().astype(bool)
t_days = ((df["ts"].to_numpy() - df["ts"].to_numpy()[0]) / DAY).astype(np.float64)
tick(f"stream assembled: {len(df):,} rows")

train: (731942, 13) | test: (262648, 12)
train window: 2026-01-01 00:00:43 -> 2026-07-15 23:58:21
test  window: 2026-07-16 00:00:21 -> 2026-09-15 22:34:38
base fraud rate: 0.017605
[    7.7s] stream assembled: 994,590 rows


In [3]:
# ============================================================================
# 03 | Missing-value handling for the categorical fields
# ============================================================================
CAT_COLS = ["merchant_category", "device_type", "location", "payment_method", "transaction_type"]

for c in ["merchant_category", "device_type", "location"]:
    df[f"{c}_was_missing"] = df[c].isna().astype(np.int8)
df["n_missing_fields"] = df[[f"{c}_was_missing" for c in
                             ["merchant_category", "device_type", "location"]]].sum(axis=1).astype(np.int8)

def dominant_attribute(frame, entity, attribute):
    """Most frequent non-null `attribute` per `entity`, learned on the historical window."""
    hist = frame.loc[frame["is_train"] == 1, [entity, attribute]].dropna()
    counts = hist.groupby([entity, attribute], observed=True).size().reset_index(name="n")
    counts = counts.sort_values("n", ascending=False).drop_duplicates(entity)
    return counts.set_index(entity)[attribute]

df["merchant_category"] = df["merchant_category"].fillna(
    df["merchant_id"].map(dominant_attribute(df, "merchant_id", "merchant_category")))
df["device_type"] = df["device_type"].fillna(
    df["device_id"].map(dominant_attribute(df, "device_id", "device_type")))
for c in CAT_COLS:
    df[c] = df[c].fillna("__NA__").astype(str)

gc.collect()
tick("categorical fields normalised")

[   10.0s] categorical fields normalised


In [4]:
# ============================================================================
# 04 | Causal feature primitives
# ============================================================================
# Every helper answers "what was true for this key strictly before the current row?". The
# stream is sorted by time, so group-wise cumulative operators are causal by construction and
# the current observation is subtracted out of every cumulative total.

def codes_of(series):
    return pd.factorize(series, sort=False)[0].astype(np.int64)

def composite(a, b):
    """Single integer key for the (a, b) pair, so pair-level history is one groupby."""
    return a.astype(np.int64) * (int(b.max()) + 1) + b.astype(np.int64)

def past_count(key_codes):
    return pd.Series(np.ones(len(key_codes))).groupby(key_codes).cumcount().to_numpy(dtype=np.float32)

def past_mean_std(key_codes, values):
    v = np.asarray(values, dtype=np.float64)
    s = pd.Series(v)
    n  = s.groupby(key_codes).cumcount().to_numpy(dtype=np.float64)
    c1 = s.groupby(key_codes).cumsum().to_numpy() - v
    c2 = pd.Series(v * v).groupby(key_codes).cumsum().to_numpy() - v * v
    safe = np.maximum(n, 1.0)
    mean_all = c1 / safe
    mean = np.where(n > 0, mean_all, np.nan)
    var  = np.where(n > 1, c2 / safe - mean_all ** 2, np.nan)
    return n.astype(np.float32), mean.astype(np.float32), np.sqrt(np.clip(var, 0.0, None)).astype(np.float32)

def past_cummax(key_codes, values):
    s = pd.Series(np.asarray(values, dtype=np.float64))
    return s.groupby(key_codes).cummax().groupby(key_codes).shift(1).to_numpy().astype(np.float32)

def past_cummin(key_codes, values):
    return pd.Series(np.asarray(values, dtype=np.float64)).groupby(key_codes).cummin().to_numpy()

def pair_past_count(key_codes, other_codes):
    tmp = pd.DataFrame({"a": key_codes, "b": other_codes})
    return tmp.groupby(["a", "b"], sort=False).cumcount().to_numpy(dtype=np.float32)

def first_occurrence(key_codes, other_codes):
    """1.0 on the row where this (key, other) pair is seen for the first time."""
    return (~pd.DataFrame({"a": key_codes, "b": other_codes}).duplicated()).to_numpy().astype(np.float64)

def past_nunique(key_codes, other_codes):
    first = first_occurrence(key_codes, other_codes)
    cum = pd.Series(first).groupby(key_codes).cumsum().to_numpy()
    return (cum - first).astype(np.float32)

def past_shift(key_codes, values, k=1):
    return pd.Series(np.asarray(values)).groupby(key_codes).shift(k)

def trailing_window_stats(key_codes, times, values, windows):
    """
    Trailing count and value-sum per key over the half-open interval [t - w, t).

    The current row is excluded by construction: within a key block the prefix sum is read at
    the row's own position, so only strictly-earlier rows contribute.
    """
    keys  = np.asarray(key_codes, dtype=np.int64)
    times = np.asarray(times, dtype=np.int64)
    vals  = np.asarray(values, dtype=np.float64)
    n = keys.shape[0]

    order = np.lexsort((times, keys))
    k, t, v = keys[order], times[order], vals[order]

    counts = {w: np.zeros(n, dtype=np.float32) for w in windows}
    sums   = {w: np.zeros(n, dtype=np.float32) for w in windows}

    starts = np.flatnonzero(np.r_[True, k[1:] != k[:-1]])
    ends   = np.r_[starts[1:], n]

    for s, e in zip(starts, ends):
        tt, vv = t[s:e], v[s:e]
        prefix = np.empty(e - s + 1, dtype=np.float64)
        prefix[0] = 0.0
        np.cumsum(vv, out=prefix[1:])
        pos = np.arange(e - s)
        for w in windows:
            left = np.searchsorted(tt, tt - w, side="left")
            counts[w][s:e] = pos - left
            sums[w][s:e]   = prefix[pos] - prefix[left]

    inverse = np.empty(n, dtype=np.int64)
    inverse[order] = np.arange(n)
    return {w: (counts[w][inverse], sums[w][inverse]) for w in windows}

def capped_log(x, cap):
    """Log-compress and cap, so a quantity that keeps growing cannot walk off the fitted range."""
    return np.log1p(np.clip(np.asarray(x, dtype=np.float64), 0.0, cap)).astype(np.float32)

tick("primitives defined")

[   10.0s] primitives defined


In [5]:
# ============================================================================
# 05 | Calendar and amount shape
# ============================================================================
ts_dt  = df["timestamp"]
amount = df["amount_bdt"].to_numpy(dtype=np.float64)

df["hour"]          = ts_dt.dt.hour.astype(np.int16)
df["minute_of_day"] = (ts_dt.dt.hour * 60 + ts_dt.dt.minute).astype(np.int16)
df["dow"]           = ts_dt.dt.dayofweek.astype(np.int8)
df["day_of_month"]  = ts_dt.dt.day.astype(np.int8)
df["is_weekend"]    = (df["dow"] >= 5).astype(np.int8)
df["hour_bucket"]   = (df["hour"] // 4).astype(np.int8)
df["is_deep_night"] = df["hour"].isin([0, 1, 2, 3, 4]).astype(np.int8)
df["is_off_hours"]  = df["hour"].isin([0, 1, 2, 3, 4, 5, 22, 23]).astype(np.int8)

ang = 2.0 * np.pi * df["minute_of_day"].to_numpy() / 1440.0
df["tod_sin"], df["tod_cos"] = np.sin(ang).astype(np.float32), np.cos(ang).astype(np.float32)
dang = 2.0 * np.pi * df["dow"].to_numpy() / 7.0
df["dow_sin"], df["dow_cos"] = np.sin(dang).astype(np.float32), np.cos(dang).astype(np.float32)

df["log_amount"]     = np.log1p(amount).astype(np.float32)
df["amount_digits"]  = np.floor(np.log10(np.maximum(amount, 1.0))).astype(np.float32)
df["is_whole_taka"]  = (np.abs(amount - np.round(amount)) < 1e-9).astype(np.int8)
df["is_round_100"]   = (np.abs(amount % 100.0) < 1e-9).astype(np.int8)
df["is_round_1000"]  = (np.abs(amount % 1000.0) < 1e-9).astype(np.int8)
df["amount_mod_100"] = (amount % 100.0).astype(np.float32)

df["account_age_days"] = df["account_age_days"].astype(np.float32)
df["log_account_age"]  = np.log1p(df["account_age_days"]).astype(np.float32)

tick("calendar and amount shape done")

[   10.5s] calendar and amount shape done


In [6]:
# ============================================================================
# 06 | Entity codes and the device identifier pool
# ============================================================================
cust_c = codes_of(df["customer_id"])
dev_c  = codes_of(df["device_id"])
merc_c = codes_of(df["merchant_id"])
loc_c  = codes_of(df["location"])
mcat_c = codes_of(df["merchant_category"])
dtyp_c = codes_of(df["device_type"])
pay_c  = codes_of(df["payment_method"])
ttyp_c = codes_of(df["transaction_type"])
hb_c   = df["hour_bucket"].to_numpy().astype(np.int64)

ts_sec  = df["ts"].to_numpy(dtype=np.int64)
log_amt = df["log_amount"].to_numpy(dtype=np.float64)
ones    = np.ones(len(df), dtype=np.float64)

df["device_ordinal"]     = df["device_id"].str.slice(1).astype(np.int64)
df["merchant_ordinal"]   = df["merchant_id"].str.slice(1).astype(np.int64)
df["log_device_ordinal"] = np.log1p(df["device_ordinal"]).astype(np.float32)

# The device identifier space has a clear structure: the settled population occupies a
# contiguous low block, and identifiers above it belong to hardware that only ever appears
# briefly. Histogram binning cannot be relied on to place a split exactly on that boundary, so
# the boundary is derived from labelled history once and passed in as an explicit indicator.
pool_ceiling = df.loc[IS_TRAIN & (df["fraud"] == 0), "device_ordinal"].max()
df["dev_outside_pool"] = (df["device_ordinal"] > pool_ceiling).astype(np.int8)
print(f"settled device-pool ceiling (from labelled history): {pool_ceiling:,}")
print(f"rows outside the pool -> train {df.loc[IS_TRAIN, 'dev_outside_pool'].mean():.4f}"
      f" | scored {df.loc[~IS_TRAIN, 'dev_outside_pool'].mean():.4f}")

tick("entity codes built")

settled device-pool ceiling (from labelled history): 15,999
rows outside the pool -> train 0.0042 | scored 0.0018
[   12.0s] entity codes built


In [7]:
# ============================================================================
# 07 | Platform traffic baseline
# ============================================================================
# Every entity volume below is expressed as a share of the platform traffic running at the
# same moment, rather than as a raw count. Raw counts inherit the growth of the stream itself
# and drift straight out of the fitted range; shares do not.
glob_win = trailing_window_stats(np.zeros(len(df), dtype=np.int64), ts_sec, ones,
                                 [HOUR, DAY, 7 * DAY, 30 * DAY])
GLOB = {name: glob_win[w][0] for w, name in
        [(HOUR, "1h"), (DAY, "24h"), (7 * DAY, "7d"), (30 * DAY, "30d")]}
for name, arr in GLOB.items():
    print(f"platform rows in trailing {name}: median {np.median(arr):,.0f}")
del glob_win
gc.collect()
tick("traffic baseline done")

platform rows in trailing 1h: median 226
platform rows in trailing 24h: median 3,803
platform rows in trailing 7d: median 26,523
platform rows in trailing 30d: median 113,248
[   12.5s] traffic baseline done


In [8]:
# ============================================================================
# 08 | Customer spending profile
# ============================================================================
n_prev, mu_log, sd_log = past_mean_std(cust_c, log_amt)
df["cust_log_amt_mean"] = mu_log
df["cust_log_amt_std"]  = sd_log
df["cust_hist_depth"]   = capped_log(n_prev, 100)          # capped: lifetime counts drift
df["amt_z_customer"]    = ((df["log_amount"] - mu_log) / (sd_log + 0.25)).astype(np.float32)

_, mu_amt, _ = past_mean_std(cust_c, amount)
df["amt_over_cust_mean"] = (amount / (mu_amt + 1.0)).astype(np.float32)

cust_max_prev = past_cummax(cust_c, amount)
df["amt_over_cust_max"] = (amount / (cust_max_prev + 1.0)).astype(np.float32)
df["exceeds_cust_max"]  = (amount > cust_max_prev).astype(np.float32)

# Exceedance profile: what fraction of this customer's own history sat above each global
# amount threshold. Scale-free, robust to the heavy right tail, and it locates the current
# amount inside the customer's personal distribution without needing an order statistic.
THRESHOLDS = np.quantile(amount[IS_TRAIN], [0.50, 0.75, 0.90, 0.95, 0.99])
for j, thr in enumerate(THRESHOLDS):
    _, rate, _ = past_mean_std(cust_c, (amount > thr).astype(np.float64))
    df[f"cust_exceed_q{j}"] = rate
    df[f"amt_vs_q{j}"] = ((amount > thr).astype(np.float32) - np.nan_to_num(rate)).astype(np.float32)
print("amount thresholds:", np.round(THRESHOLDS, 2))

cust_win = trailing_window_stats(cust_c, ts_sec, amount, [300, 900, HOUR, 6 * HOUR, DAY, 7 * DAY, 30 * DAY])
for w, name in [(300, "5m"), (900, "15m"), (HOUR, "1h"), (6 * HOUR, "6h"),
                (DAY, "24h"), (7 * DAY, "7d"), (30 * DAY, "30d")]:
    c, s = cust_win[w]
    df[f"cust_cnt_{name}"] = c                              # per-customer velocity stays small
    df[f"cust_avg_amt_{name}"] = (s / np.maximum(c, 1.0)).astype(np.float32)
    if name in ("24h", "7d", "30d"):
        df[f"amt_over_{name}_spend"] = (amount / (s + 1.0)).astype(np.float32)

df["cust_burst_1h_24h"]   = (df["cust_cnt_1h"]  / (df["cust_cnt_24h"] + 1.0)).astype(np.float32)
df["cust_burst_24h_7d"]   = (df["cust_cnt_24h"] / (df["cust_cnt_7d"]  + 1.0)).astype(np.float32)
df["cust_burst_7d_30d"]   = (df["cust_cnt_7d"]  / (df["cust_cnt_30d"] + 1.0)).astype(np.float32)
df["cust_recent_vs_long"] = (df["cust_avg_amt_7d"] / (mu_amt + 1.0)).astype(np.float32)
df["cust_recent_vs_mid"]  = (df["cust_avg_amt_24h"] / (df["cust_avg_amt_30d"] + 1.0)).astype(np.float32)

del cust_win
gc.collect()
tick("customer spending profile done")

amount thresholds: [ 492.38 1019.2  1992.65 3031.24 7810.6 ]
[   17.5s] customer spending profile done


In [9]:
# ============================================================================
# 09 | Customer tempo
# ============================================================================
prev_ts  = past_shift(cust_c, ts_sec, 1)
prev_ts2 = past_shift(cust_c, ts_sec, 2)
prev_ts5 = past_shift(cust_c, ts_sec, 5)

gap = (ts_sec - prev_ts).astype(np.float64)
df["cust_log_gap"] = np.log1p(gap.clip(lower=0)).astype(np.float32)
df["cust_gap_2"]   = np.log1p((ts_sec - prev_ts2).clip(lower=0)).astype(np.float32)
df["cust_gap_5"]   = np.log1p((ts_sec - prev_ts5).clip(lower=0)).astype(np.float32)

first_ts = past_cummin(cust_c, ts_sec)
tenure   = ts_sec - first_ts
# Tenure is capped: it grows one-for-one with calendar time and would otherwise leave the
# fitted range entirely by the end of the scoring window.
# Only the *young* end of tenure carries signal. The continuous value is a pure restatement
# of calendar time for any long-lived account, so it saturates in the scored period and is
# represented by short-horizon indicators instead.
df["cust_is_fresh"]    = (tenure < 7 * DAY).astype(np.int8)
df["cust_age_lt_30d"]  = (tenure < 30 * DAY).astype(np.int8)
mean_gap = tenure / np.maximum(n_prev, 1.0)
df["cust_mean_gap"]    = np.log1p(mean_gap).astype(np.float32)
df["cust_gap_ratio"]   = (gap.to_numpy() / (mean_gap + 1.0)).astype(np.float32)
df["cust_txn_per_day"] = (n_prev / (tenure / DAY + 1.0)).astype(np.float32)

first_age = pd.Series(df["account_age_days"].to_numpy()).groupby(cust_c).transform("first").to_numpy()
drift = df["account_age_days"].to_numpy() - first_age - tenure / DAY
df["account_age_drift"] = np.clip(drift, -30, 30).astype(np.float32)
df["age_over_tenure"]   = np.clip(df["account_age_days"].to_numpy() / (tenure / DAY + 1.0),
                                  0, 5000).astype(np.float32)

sin_col = df["tod_sin"].to_numpy(dtype=np.float64)
cos_col = df["tod_cos"].to_numpy(dtype=np.float64)
_, mu_sin, _ = past_mean_std(cust_c, sin_col)
_, mu_cos, _ = past_mean_std(cust_c, cos_col)
mu_sin, mu_cos = np.nan_to_num(mu_sin.astype(np.float64)), np.nan_to_num(mu_cos.astype(np.float64))
resultant = np.sqrt(mu_sin ** 2 + mu_cos ** 2)
cos_delta = np.where(resultant > EPS, (sin_col * mu_sin + cos_col * mu_cos) / np.maximum(resultant, EPS), np.nan)
df["cust_hour_concentration"] = np.where(n_prev > 0, resultant, np.nan).astype(np.float32)
df["cust_hour_deviation"]     = np.arccos(np.clip(cos_delta, -1.0, 1.0)).astype(np.float32)

# How nocturnal is this customer normally, and is this transaction out of character for them?
_, night_rate, _ = past_mean_std(cust_c, df["is_deep_night"].to_numpy(dtype=np.float64))
df["cust_night_rate"] = night_rate
df["night_off_profile"] = (df["is_deep_night"].to_numpy() - np.nan_to_num(night_rate)).astype(np.float32)

tick("customer tempo done")

[   18.9s] customer tempo done


In [10]:
# ============================================================================
# 10 | Familiarity and per-pair recency
# ============================================================================
# Beyond "has this customer used this entity before", how long ago was the last time? A
# dormant pairing waking up behaves differently from an actively used one.
familiarity = [("device", dev_c), ("merchant", merc_c), ("loc", loc_c),
               ("mcat", mcat_c), ("dtype", dtyp_c), ("pay", pay_c),
               ("ttype", ttyp_c), ("hourband", hb_c)]

for name, codes in familiarity:
    pc = pair_past_count(cust_c, codes)
    df[f"cust_{name}_is_new"] = (pc == 0).astype(np.int8)
    df[f"cust_{name}_share"]  = (pc / np.maximum(n_prev, 1.0)).astype(np.float32)
    df[f"cust_{name}_depth"]  = capped_log(pc, 100)
    nu = past_nunique(cust_c, codes)
    df[f"cust_n_{name}"]      = capped_log(nu, 50)
    df[f"cust_{name}_div"]    = (nu / np.maximum(n_prev, 1.0)).astype(np.float32)

for name, codes in [("device", dev_c), ("merchant", merc_c), ("loc", loc_c)]:
    pk = composite(cust_c, codes)
    last_seen = past_shift(pk, ts_sec, 1)
    df[f"cust_{name}_recency"] = np.log1p((ts_sec - last_seen).clip(lower=0)).astype(np.float32)

# Has this customer transacted this exact amount before? Repeated amounts are a signature of
# scripted rather than organic activity.
amt_code = codes_of(pd.Series(np.round(amount, 2)))
df["cust_same_amt_seen"] = capped_log(pair_past_count(cust_c, amt_code), 50)

# Amount normality inside the customer's own behaviour *for this category of merchant*.
cm_key = composite(cust_c, mcat_c)
_, cm_mu, cm_sd = past_mean_std(cm_key, log_amt)
df["amt_z_cust_mcat"] = ((df["log_amount"] - cm_mu) / (cm_sd + 0.25)).astype(np.float32)

df["new_device_x_history"] = (df["cust_device_is_new"]   * np.log1p(n_prev)).astype(np.float32)
df["new_loc_x_history"]    = (df["cust_loc_is_new"]      * np.log1p(n_prev)).astype(np.float32)
df["new_merch_x_history"]  = (df["cust_merchant_is_new"] * np.log1p(n_prev)).astype(np.float32)
df["novelty_score"] = (df["cust_device_is_new"] + df["cust_loc_is_new"]
                       + df["cust_merchant_is_new"] + df["cust_hourband_is_new"]).astype(np.float32)
df["novelty_x_night"] = (df["novelty_score"] * df["is_deep_night"]).astype(np.float32)
df["novelty_x_amtz"]  = (df["novelty_score"] * df["amt_z_customer"].fillna(0)).astype(np.float32)

tick("familiarity and recency done")

[   24.7s] familiarity and recency done


In [11]:
# ============================================================================
# 11 | Relationship structure - who is acquiring whom, and how fast
# ============================================================================
# A device quietly serving the same account for months is a different object from one that
# picked up nine new accounts this week. Counting *first-time pairings inside a trailing
# window* captures that directly, and because the window is bounded the feature does not drift.
new_dev_cust = first_occurrence(dev_c, cust_c)
w = trailing_window_stats(dev_c, ts_sec, new_dev_cust, [DAY, 7 * DAY, 30 * DAY])
df["dev_new_cust_24h"] = w[DAY][1]
df["dev_new_cust_7d"]  = w[7 * DAY][1]
df["dev_new_cust_30d"] = w[30 * DAY][1]

new_merch_cust = first_occurrence(merc_c, cust_c)
w = trailing_window_stats(merc_c, ts_sec, new_merch_cust, [HOUR, DAY])
df["merch_new_cust_1h"]  = w[HOUR][1]
df["merch_new_cust_24h"] = w[DAY][1]

new_cust_dev = first_occurrence(cust_c, dev_c)
w = trailing_window_stats(cust_c, ts_sec, new_cust_dev, [DAY, 7 * DAY])
df["cust_new_dev_24h"] = w[DAY][1]
df["cust_new_dev_7d"]  = w[7 * DAY][1]

new_cust_loc = first_occurrence(cust_c, loc_c)
w = trailing_window_stats(cust_c, ts_sec, new_cust_loc, [DAY, 7 * DAY])
df["cust_new_loc_24h"] = w[DAY][1]
df["cust_new_loc_7d"]  = w[7 * DAY][1]

new_cust_merch = first_occurrence(cust_c, merc_c)
w = trailing_window_stats(cust_c, ts_sec, new_cust_merch, [DAY, 7 * DAY])
df["cust_new_merch_7d"] = w[7 * DAY][1]

# Acquisition rate relative to the device's own throughput: ten new accounts on a device
# handling a thousand transactions a week is ordinary; on a device handling twelve, it is not.
del w
gc.collect()
tick("relationship structure done")

[   30.9s] relationship structure done


In [12]:
# ============================================================================
# 12 | Device behaviour, expressed as rates and shares
# ============================================================================
dev_prev = past_count(dev_c)
df["dev_is_first_use"] = (dev_prev == 0).astype(np.int8)

dev_first_ts = past_cummin(dev_c, ts_sec)
dev_age_days = (ts_sec - dev_first_ts) / DAY
df["dev_age_lt_1d"]   = (dev_age_days < 1).astype(np.int8)
df["dev_age_lt_7d"]   = (dev_age_days < 7).astype(np.int8)
df["dev_age_lt_30d"]  = (dev_age_days < 30).astype(np.int8)
df["dev_txn_per_day"] = (dev_prev / (dev_age_days + 1.0)).astype(np.float32)

dev_nc = past_nunique(dev_c, cust_c)
df["dev_cust_per_day"] = (dev_nc / (dev_age_days + 1.0)).astype(np.float32)
df["dev_cust_per_txn"] = (dev_nc / np.maximum(dev_prev, 1.0)).astype(np.float32)
df["dev_is_shared"]    = (dev_nc > 1).astype(np.int8)
df["dev_n_cust_capped"] = capped_log(dev_nc, 40)
df["dev_n_loc_capped"]  = capped_log(past_nunique(dev_c, loc_c), 45)

dev_win = trailing_window_stats(dev_c, ts_sec, amount, [HOUR, DAY, 7 * DAY])
for w_, name in [(HOUR, "1h"), (DAY, "24h"), (7 * DAY, "7d")]:
    c, s = dev_win[w_]
    df[f"dev_cnt_{name}"]     = c
    df[f"dev_share_{name}"]   = (c / (GLOB[name] + 1.0)).astype(np.float32)
    df[f"dev_avg_amt_{name}"] = (s / np.maximum(c, 1.0)).astype(np.float32)
df["dev_burst_1h_7d"]  = (df["dev_cnt_1h"] / (df["dev_cnt_7d"] / 168.0 + 1.0)).astype(np.float32)
df["dev_acq_intensity"] = (df["dev_new_cust_7d"] / (df["dev_cnt_7d"] + 1.0)).astype(np.float32)

_, dev_mu, dev_sd = past_mean_std(dev_c, log_amt)
df["amt_z_device"] = ((df["log_amount"] - dev_mu) / (dev_sd + 0.25)).astype(np.float32)
df["cold_device_amount"] = (df["log_amount"] / (np.log1p(dev_prev) + 1.0)).astype(np.float32)

del dev_win
gc.collect()
tick("device behaviour done")

[   32.7s] device behaviour done


In [13]:
# ============================================================================
# 13 | Merchant behaviour and population baselines
# ============================================================================
merc_prev = past_count(merc_c)
df["merch_is_first_use"] = (merc_prev == 0).astype(np.int8)

merch_first_ts = past_cummin(merc_c, ts_sec)
merch_age_days = (ts_sec - merch_first_ts) / DAY
df["merch_age_lt_7d"]   = (merch_age_days < 7).astype(np.int8)
df["merch_age_lt_30d"]  = (merch_age_days < 30).astype(np.int8)
df["merch_txn_per_day"] = (merc_prev / (merch_age_days + 1.0)).astype(np.float32)

merch_nc = past_nunique(merc_c, cust_c)
df["merch_cust_per_day"] = (merch_nc / (merch_age_days + 1.0)).astype(np.float32)
df["merch_repeat_rate"]  = (merc_prev / np.maximum(merch_nc, 1.0)).astype(np.float32)
df["merch_dev_per_cust"] = (past_nunique(merc_c, dev_c) / np.maximum(merch_nc, 1.0)).astype(np.float32)

merc_win = trailing_window_stats(merc_c, ts_sec, amount, [HOUR, DAY, 7 * DAY, 30 * DAY])
for w_, name in [(HOUR, "1h"), (DAY, "24h"), (7 * DAY, "7d"), (30 * DAY, "30d")]:
    c, s = merc_win[w_]
    df[f"merch_share_{name}"]   = (c / (GLOB[name] + 1.0)).astype(np.float32)
    df[f"merch_avg_amt_{name}"] = (s / np.maximum(c, 1.0)).astype(np.float32)
    if name in ("1h", "24h", "7d", "30d"):
        df[f"_mcnt_{name}"] = c

# Surge is a ratio of the merchant's own rates, so it is unaffected by platform growth.
df["merch_surge_1h"]  = (df["_mcnt_1h"]  / (df["_mcnt_7d"] / 168.0 + 1.0)).astype(np.float32)
df["merch_surge_24h"] = (df["_mcnt_24h"] / (df["_mcnt_30d"] / 30.0 + 1.0)).astype(np.float32)
df["merch_acq_intensity"] = (df["merch_new_cust_24h"] / (df["_mcnt_24h"] + 1.0)).astype(np.float32)
df.drop(columns=[c for c in df.columns if c.startswith("_mcnt_")], inplace=True)

_, merc_mu, merc_sd = past_mean_std(merc_c, log_amt)
df["amt_z_merchant"] = ((df["log_amount"] - merc_mu) / (merc_sd + 0.25)).astype(np.float32)

for tag, codes in [("mcat", mcat_c), ("dtype", dtyp_c), ("pay", pay_c), ("ttype", ttyp_c), ("loc", loc_c)]:
    _, mu_g, sd_g = past_mean_std(codes, log_amt)
    df[f"amt_z_{tag}"] = ((df["log_amount"] - mu_g) / (sd_g + 0.25)).astype(np.float32)

_, glob_mu, glob_sd = past_mean_std(np.zeros(len(df), dtype=np.int64), log_amt)
df["amt_z_global"] = ((df["log_amount"] - glob_mu) / (glob_sd + 0.25)).astype(np.float32)

# Amount normality conditional on the hour of day, which matters because the overnight
# population is small and behaves differently from the daytime population.
lh_key = composite(loc_c, hb_c)
_, lh_mu, lh_sd = past_mean_std(lh_key, log_amt)
df["amt_z_loc_hour"] = ((df["log_amount"] - lh_mu) / (lh_sd + 0.25)).astype(np.float32)

del merc_win
gc.collect()
tick("merchant and population baselines done")

[   36.4s] merchant and population baselines done


In [14]:
# ============================================================================
# 14 | Movement plausibility
# ============================================================================
prev_loc  = past_shift(cust_c, loc_c, 1)
prev_loc2 = past_shift(cust_c, loc_c, 2)
prev_loc3 = past_shift(cust_c, loc_c, 3)

loc_changed = (prev_loc.notna() & (prev_loc.to_numpy() != loc_c)).to_numpy().astype(np.float32)
df["loc_changed"] = loc_changed

gap_hours = np.maximum(np.nan_to_num(gap.to_numpy(), nan=np.inf), 0.0) / 3600.0
df["loc_move_speed"]  = (loc_changed / (gap_hours + 0.05)).astype(np.float32)
df["impossible_move"] = ((loc_changed > 0) & (gap_hours < 0.5)).astype(np.int8)
df["loc_switches_3"]  = (
    (prev_loc.notna()  & (prev_loc.to_numpy()  != loc_c)).astype(np.int8)
    + (prev_loc2.notna() & (prev_loc2.to_numpy() != loc_c)).astype(np.int8)
    + (prev_loc3.notna() & (prev_loc3.to_numpy() != loc_c)).astype(np.int8)
).astype(np.int8)

prev_dev = past_shift(cust_c, dev_c, 1)
df["device_changed"] = (prev_dev.notna() & (prev_dev.to_numpy() != dev_c)).astype(np.int8)
df["device_switch_speed"] = (df["device_changed"].to_numpy() / (gap_hours + 0.05)).astype(np.float32)

prev_amt = past_shift(cust_c, amount, 1).to_numpy()
df["amt_jump_ratio"] = np.clip(amount / (np.nan_to_num(prev_amt, nan=0.0) + 1.0), 0, 1e5).astype(np.float32)
df["log_amt_delta"]  = (df["log_amount"].to_numpy() - np.log1p(prev_amt)).astype(np.float32)

loc_win = trailing_window_stats(loc_c, ts_sec, amount, [HOUR, DAY])
for w_, name in [(HOUR, "1h"), (DAY, "24h")]:
    c, s = loc_win[w_]
    df[f"loc_share_{name}"]   = (c / (GLOB[name] + 1.0)).astype(np.float32)
    df[f"loc_avg_amt_{name}"] = (s / np.maximum(c, 1.0)).astype(np.float32)

del loc_win
gc.collect()
tick("movement features done")

[   37.2s] movement features done


In [15]:
# ============================================================================
# 14b | Additional behavioural families
# ============================================================================
# Five families the earlier passes do not cover. Each was measured to carry real weight in
# parallel experiments on this dataset, and each is computed with the same causal
# primitives as everything above - strictly-earlier rows only, current row excluded.
#
#   * personal rate ratios - this hour's activity against the customer's OWN weekly rate,
#     which is different from a share of platform traffic and was the second-highest gain
#     feature in a parallel run;
#   * concentration (Herfindahl) - how repetitive a history is. An unfamiliar device means
#     far more against a settled profile than against a scattered one;
#   * arrival-process acceleration and dispersion - a collapsing gap is the burst signature,
#     and a gap is only unusual relative to how regular the customer normally is;
#   * cross-entity pairings - a device meeting a merchant for the first time is a fact about
#     the network rather than about the account;
#   * a peer baseline - what a comparable account, in the same place, doing the same kind of
#     transaction, normally spends. It gives a reference for customers whose own history is
#     too thin to produce a usable z-score.

amount_v = df["amount_bdt"].to_numpy(dtype=np.float64)
n_prev_v = past_count(cust_c)

# --- personal rate ratios ---------------------------------------------------
_cw = trailing_window_stats(cust_c, ts_sec, ones, [HOUR, DAY, 7 * DAY, 30 * DAY])
_c1h, _c24h, _c7d, _c30d = (_cw[HOUR][0], _cw[DAY][0], _cw[7 * DAY][0], _cw[30 * DAY][0])
df["cust_rate_ratio_1h"]  = (_c1h  / (_c7d / 168.0 + 0.5)).astype(np.float32)
df["cust_rate_ratio_24h"] = (_c24h / (_c30d / 30.0 + 0.5)).astype(np.float32)

_dw = trailing_window_stats(dev_c, ts_sec, ones, [HOUR, 7 * DAY])
df["dev_rate_ratio_1h"] = (_dw[HOUR][0] / (_dw[7 * DAY][0] / 168.0 + 0.5)).astype(np.float32)

# --- concentration ----------------------------------------------------------
# Raising a value's count from c to c+1 raises the sum of squared counts by 2c+1, so the
# Herfindahl index over strictly earlier rows is a single cumulative sum - exact, not
# approximated, and one pass per key.
def past_hhi(key_codes, other_codes, depth):
    inc = 2.0 * pair_past_count(key_codes, other_codes) + 1.0
    sumsq = pd.Series(inc).groupby(key_codes).cumsum().to_numpy() - inc
    return (sumsq / np.maximum(depth, 1.0) ** 2).astype(np.float32)

for nm, cd in [("device", dev_c), ("merchant", merc_c), ("loc", loc_c), ("hourband", hb_c)]:
    df[f"cust_{nm}_hhi"] = past_hhi(cust_c, cd, n_prev_v)
df["dev_cust_hhi"] = past_hhi(dev_c, cust_c, past_count(dev_c))

df["new_device_x_hhi"] = (df["cust_device_is_new"].to_numpy() * df["cust_device_hhi"].to_numpy()).astype(np.float32)
df["new_loc_x_hhi"]    = (df["cust_loc_is_new"].to_numpy()    * df["cust_loc_hhi"].to_numpy()).astype(np.float32)

# --- arrival process --------------------------------------------------------
_prev_ts = past_shift(cust_c, ts_sec, 1)
_gap = (ts_sec - _prev_ts).astype(np.float64)
_log_gap = np.log1p(np.nan_to_num(_gap.to_numpy(), nan=0.0).clip(min=0))
_prev_log_gap = past_shift(cust_c, _log_gap, 1).to_numpy()
df["cust_gap_accel"] = (_log_gap - np.nan_to_num(_prev_log_gap, nan=float(_log_gap.mean()))).astype(np.float32)
_, _gmu, _gsd = past_mean_std(cust_c, _log_gap)
df["cust_gap_z"]  = ((_log_gap - _gmu) / (_gsd + 0.5)).astype(np.float32)
df["cust_gap_sd"] = _gsd

# --- cross-entity pairings --------------------------------------------------
for a_nm, a_cd, b_nm, b_cd in [("dev", dev_c, "merch", merc_c),
                               ("dev", dev_c, "loc", loc_c),
                               ("merch", merc_c, "loc", loc_c)]:
    _pc = pair_past_count(a_cd, b_cd)
    df[f"{a_nm}_{b_nm}_pair_depth"]  = capped_log(_pc, 5000)
    df[f"{a_nm}_{b_nm}_pair_is_new"] = (_pc == 0).astype(np.int8)

# --- peer baseline ----------------------------------------------------------
_age_bucket = np.digitize(df["account_age_days"].to_numpy(), [30, 90, 180, 365, 730, 1460])
_peer_c = codes_of(pd.Series(composite(composite(_age_bucket.astype(np.int64), loc_c), ttyp_c)))
_, _pmu, _psd = past_mean_std(_peer_c, log_amt)
df["amt_z_peer"]   = ((df["log_amount"].to_numpy() - _pmu) / (_psd + 0.25)).astype(np.float32)
df["peer_support"] = capped_log(past_count(_peer_c), 200000)

# --- merchant behaviour shift ----------------------------------------------
_moh = trailing_window_stats(merc_c, ts_sec, df["is_off_hours"].to_numpy().astype(np.float64),
                             [DAY, 7 * DAY])
_mw = trailing_window_stats(merc_c, ts_sec, ones, [DAY, 7 * DAY])
_r24 = _moh[DAY][1] / (_mw[DAY][0] + 1.0)
_r7d = _moh[7 * DAY][1] / (_mw[7 * DAY][0] + 1.0)
df["merch_offhour_rate_24h"] = _r24.astype(np.float32)
df["merch_offhour_shift"]    = (_r24 - _r7d).astype(np.float32)

del _cw, _dw, _moh, _mw, _prev_ts, _gap, _log_gap, _prev_log_gap
gc.collect()
tick("additional behavioural families done")


[   43.1s] additional behavioural families done


In [16]:
# ============================================================================
# 15 | Target encodings - expanding, decayed, and now per-entity
# ============================================================================
# Two views of entity risk. The expanding encoding is the long-run rate; the decayed encoding
# weights recent labelled evidence far more heavily, so an entity whose risk profile changed
# is reflected quickly. Both read labelled rows strictly earlier than the row being encoded,
# and both are frozen at the end of history before being applied to the scoring window -
# where the decay additionally pulls the value back toward the base rate the further out we
# go, which is the correct behaviour when the evidence is stale.
#
# Two key groups, differing only in how much prior weight they carry:
#
#   moderate cardinality (loc, merchant, and the two hour crosses) keep the heavy prior - a
#     busy merchant accumulates thousands of labelled rows and the encoding converges anyway;
#
#   high cardinality (customer, device) get a lighter prior. A customer carries a median of
#     ten prior rows, so a prior weight of 50 would flatten the encoding into a constant.
#     These two are the largest gap in the original feature set: both identifiers are dropped
#     from the model entirely, so their fraud history has no other route in.
#
# A support column accompanies each encoding, so the model can tell a rate backed by six
# observations from the same rate backed by six hundred.
y_tr = df.loc[IS_TRAIN, "fraud"].to_numpy(dtype=np.float64)
prior = float(y_tr.mean())
print("prior fraud rate:", round(prior, 6))

TE_KEYS = {"loc": (loc_c, TE_SMOOTH), "merchant": (merc_c, TE_SMOOTH),
           "loc_hour": (lh_key, TE_SMOOTH), "mcat_hour": (composite(mcat_c, hb_c), TE_SMOOTH),
           "customer": (cust_c, TE_SMOOTH_HI), "device": (dev_c, TE_SMOOTH_HI)}
NEW_TE_COLS = []

t_days_tr = t_days[IS_TRAIN]
w_decay_all = np.exp(t_days / TE_TAU)
w_decay_tr  = w_decay_all[IS_TRAIN]

for name, (codes, smooth) in TE_KEYS.items():
    k_tr = codes[IS_TRAIN]

    # Expanding view.
    s = pd.Series(y_tr)
    n_before = s.groupby(k_tr).cumcount().to_numpy(dtype=np.float64)
    p_before = s.groupby(k_tr).cumsum().to_numpy() - y_tr
    exp_out = np.full(len(df), np.nan, dtype=np.float64)
    sup_out = np.full(len(df), np.nan, dtype=np.float64)
    exp_out[IS_TRAIN] = (p_before + prior * smooth) / (n_before + smooth)
    sup_out[IS_TRAIN] = n_before
    agg = pd.DataFrame({"k": k_tr, "y": y_tr}).groupby("k")["y"].agg(["sum", "count"])
    frozen = (agg["sum"] + prior * smooth) / (agg["count"] + smooth)
    exp_out[~IS_TRAIN] = pd.Series(codes[~IS_TRAIN]).map(frozen).fillna(prior).to_numpy()
    sup_out[~IS_TRAIN] = pd.Series(codes[~IS_TRAIN]).map(agg["count"]).fillna(0.0).to_numpy()
    df[f"te_{name}"]  = exp_out.astype(np.float32)
    df[f"ten_{name}"] = np.log1p(sup_out).astype(np.float32)

    # Decayed view. Prefix sums of y*exp(t/tau) and exp(t/tau) over prior labelled rows; the
    # common exp(t/tau) factor cancels in the ratio, leaving an exponentially weighted rate.
    A = pd.Series(y_tr * w_decay_tr).groupby(k_tr).cumsum().to_numpy() - y_tr * w_decay_tr
    B = pd.Series(w_decay_tr).groupby(k_tr).cumsum().to_numpy() - w_decay_tr
    dec_out = np.full(len(df), np.nan, dtype=np.float64)
    scale_tr = np.exp(-t_days_tr / TE_TAU)
    dec_out[IS_TRAIN] = (A * scale_tr + prior * smooth) / (B * scale_tr + smooth)
    fin = pd.DataFrame({"k": k_tr, "a": y_tr * w_decay_tr, "b": w_decay_tr}).groupby("k").sum()
    a_map = pd.Series(codes[~IS_TRAIN]).map(fin["a"]).fillna(0.0).to_numpy()
    b_map = pd.Series(codes[~IS_TRAIN]).map(fin["b"]).fillna(0.0).to_numpy()
    scale_te = np.exp(-t_days[~IS_TRAIN] / TE_TAU)
    dec_out[~IS_TRAIN] = (a_map * scale_te + prior * smooth) / (b_map * scale_te + smooth)
    df[f"ted_{name}"] = dec_out.astype(np.float32)

    if name in ("customer", "device"):
        NEW_TE_COLS += [f"te_{name}", f"ten_{name}", f"ted_{name}"]

print("per-entity encodings added:", NEW_TE_COLS)
del y_tr, w_decay_all, w_decay_tr
gc.collect()
tick("target encodings done")


prior fraud rate: 0.017605
per-entity encodings added: ['te_customer', 'ten_customer', 'ted_customer', 'te_device', 'ten_device', 'ted_device']
[   44.8s] target encodings done


In [17]:
# ============================================================================
# 16 | Assemble the design matrix
# ============================================================================
DROP = {"transaction_id", "customer_id", "merchant_id", "device_id",
        "timestamp", "ts", "fraud", "is_train"}
FEATURES = [c for c in df.columns if c not in DROP]

for c in CAT_COLS:
    df[c] = df[c].astype("category")
for c in FEATURES:
    if c in CAT_COLS:
        continue
    if df[c].dtype != np.float32:
        df[c] = df[c].astype(np.float32)
    df[c] = df[c].replace([np.inf, -np.inf], np.nan)

print(f"feature count: {len(FEATURES)}")
X_all, y_all, ts_all = df[FEATURES], df["fraud"], df["timestamp"]
gc.collect()
tick("design matrix assembled")

feature count: 250
[   49.2s] design matrix assembled


In [18]:
# ============================================================================
# 17 | Validation schedule - two folds
# ============================================================================
# `w4` is gone. It scores exactly the same 82,586 rows as `recent`, differing only in that its
# fitting window runs a day before the block instead of thirty days before. That makes it the
# easier and less realistic of the pair, and running both cost a third of every model cell for
# a second opinion on the same rows.
#
#   horizon  fit to 13 May, score the next 62 days in one block. Right forecasting DISTANCE,
#            three times the positives of the other fold, but roughly two-thirds of its
#            content predates the behaviour shift around 24 June.
#   recent   fit to 25 May, score 24 Jun -> 15 Jul.  Right CONTENT at a realistic distance.
#
# `recent` leads the weighting because it is the only block that matches the scoring window on
# both content and distance; `horizon` is kept for its far lower noise.
folds = []

h_end   = TRAIN_END
h_start = h_end - pd.Timedelta(days=HORIZON_DAYS)
folds.append((h_start - pd.Timedelta(days=EMBARGO_DAYS), h_start, h_end))

r_end   = TRAIN_END
r_start = r_end - pd.Timedelta(days=VALID_DAYS)
folds.append((r_start - pd.Timedelta(days=FAR_EMBARGO), r_start, r_end))

FOLD_NAMES = ["horizon", "recent"]
HORIZON_IX, RECENT_IX = 0, 1
EARLIEST_FIT_END = min(f[0] for f in folds)

FIT_OK = IS_TRAIN & (ts_all >= STREAM_START + pd.Timedelta(days=WARMUP_DAYS)).to_numpy()
print(f"warm-up rows excluded from fitting: {(IS_TRAIN & ~FIT_OK).sum():,}\n")

fold_masks, FOLD_CUTOFFS = [], []
for (fit_end, v_start, v_end), nm in zip(folds, FOLD_NAMES):
    tr_m = FIT_OK & (ts_all <= fit_end).to_numpy()
    va_m = IS_TRAIN & (ts_all > v_start).to_numpy() & (ts_all <= v_end).to_numpy()
    fold_masks.append((tr_m, va_m))
    FOLD_CUTOFFS.append(int(pd.Timestamp(fit_end).timestamp()))
    print(f"{nm:>8}  fit <= {fit_end:%Y-%m-%d}  score {v_start:%Y-%m-%d} -> {v_end:%Y-%m-%d}"
          f"  |  {tr_m.sum():>7,} fit / {va_m.sum():>6,} scored  |  rate {y_all[va_m].mean():.4f}")

SEL_WEIGHTS = np.array([0.35, 0.65])
assert len(SEL_WEIGHTS) == len(fold_masks)

def selection_score(aps):
    return float(np.dot(SEL_WEIGHTS, np.asarray(aps)))

def to_rank(p):
    return rankdata(p) / len(p)

tick("validation schedule built")


warm-up rows excluded from fitting: 107,721

 horizon  fit <= 2026-05-13  score 2026-05-14 -> 2026-07-15  |  380,245 fit / 240,065 scored  |  rate 0.0168
  recent  fit <= 2026-05-25  score 2026-06-24 -> 2026-07-15  |  425,622 fit / 82,762 scored  |  rate 0.0161
[   49.3s] validation schedule built


In [19]:
# ============================================================================
# 18 | Feature ranking
# ============================================================================
# The adversarial drift probe is gone. It ran in two consecutive versions, reported an AUC of
# 1.0000 both times, and pruned zero features both times - the rule needs a feature that both
# drifts hard and earns little, and no feature here does both. What it was actually used for
# was a gain ranking, and one short high-rate fit supplies that far more cheaply.
import lightgbm as lgb

LGB_PARAMS = dict(
    objective="binary", metric="average_precision", learning_rate=0.03,
    num_leaves=127, min_data_in_leaf=50, feature_fraction=0.65,
    bagging_fraction=0.85, bagging_freq=1, lambda_l1=0.5, lambda_l2=5.0,
    max_bin=511, max_cat_threshold=48, cat_smooth=20.0,
    verbosity=-1, n_jobs=-1, seed=SEEDS[0],
)
MAX_ROUNDS, EARLY_STOP = 4000, 200

tr_r, va_r = fold_masks[RECENT_IX]
ALL_CATS = [c for c in CAT_COLS if c in FEATURES]
rank_params = dict(LGB_PARAMS); rank_params["learning_rate"] = 0.12
m_rank = lgb.train(rank_params,
                   lgb.Dataset(X_all.loc[tr_r, FEATURES], label=y_all[tr_r],
                               categorical_feature=ALL_CATS),
                   num_boost_round=300)
gain_share = pd.Series(m_rank.feature_importance("gain"), index=FEATURES)
gain_share = gain_share / max(gain_share.sum(), 1e-9)
del m_rank; gc.collect()
KEPT = list(FEATURES)
print("top 20 by gain")
print(gain_share.sort_values(ascending=False).head(20).round(5).to_string())
print("\nwhere the per-entity encodings rank:")
for c in NEW_TE_COLS:
    r = int((gain_share > gain_share.get(c, 0)).sum()) + 1
    print(f"  {c:<16} gain {gain_share.get(c, 0):.5f}   rank {r} of {len(FEATURES)}")
tick("feature ranking done")


top 20 by gain
dev_outside_pool       0.23389
cust_rate_ratio_1h     0.12451
amt_vs_q4              0.07768
merch_surge_24h        0.06714
amt_over_cust_mean     0.04110
ted_device             0.02368
cust_loc_recency       0.02144
cust_device_share      0.02107
ted_merchant           0.01930
amt_jump_ratio         0.01890
location               0.01580
te_device              0.01292
cust_device_recency    0.01212
te_customer            0.00899
amt_z_customer         0.00826
amt_over_cust_max      0.00816
device_ordinal         0.00716
ted_customer           0.00711
amt_vs_q3              0.00642
cust_gap_ratio         0.00626

where the per-entity encodings rank:
  te_customer      gain 0.00899   rank 14 of 250
  ten_customer     gain 0.00038   rank 168 of 250
  ted_customer     gain 0.00711   rank 18 of 250
  te_device        gain 0.01292   rank 12 of 250
  ten_device       gain 0.00112   rank 112 of 250
  ted_device       gain 0.02368   rank 6 of 250
[  176.4s] feature ranking done


In [20]:
# ============================================================================
# 19b | Mine the rule-shaped part of the fraud into explicit indicators
# ============================================================================
# A segment scan on this dataset found single thresholds isolating fraud at 60-70x the base
# rate. A boosted tree can find those, but only to its binning resolution and only by spending
# splits on them; handing over the exact cut is cheaper and sharper.
#
# The thresholds are fitted on labelled rows ending BEFORE the earliest fold's fitting window
# closes, so no validation block contributes a label to a rule it is later scored against, and
# the same static columns serve validation and scoring alike.
rule_base = FIT_OK & (ts_all <= EARLIEST_FIT_END).to_numpy()
yb = y_all[rule_base].to_numpy()
base_rate = float(yb.mean())
print(f"mining rules on {rule_base.sum():,} rows up to {EARLIEST_FIT_END:%Y-%m-%d} "
      f"(base rate {base_rate:.4f})")

cands = [f for f in gain_share.sort_values(ascending=False).head(40).index
         if f not in CAT_COLS]
found = []
for f in cands:
    v = X_all.loc[rule_base, f].to_numpy(dtype=np.float64)
    ok = np.isfinite(v)
    if int(ok.sum()) < 5000:
        continue
    best = None
    for q, side in [(0.9995, "hi"), (0.999, "hi"), (0.995, "hi"), (0.99, "hi"),
                    (0.0005, "lo"), (0.001, "lo"), (0.005, "lo"), (0.01, "lo")]:
        thr = float(np.quantile(v[ok], q))
        sel = ((v >= thr) if side == "hi" else (v <= thr)) & ok
        n = int(sel.sum())
        if n < 100:
            continue
        prec = float(yb[sel].mean())
        rec = float(yb[sel].sum() / max(yb.sum(), 1.0))
        if prec >= RULE_MIN_PREC and rec >= RULE_MIN_REC:
            if best is None or prec > best[3]:
                best = (f, side, thr, prec, rec, n)
    if best is not None:
        found.append(best)

found.sort(key=lambda r: -r[3])
found = found[:RULE_MAX]
RULE_COLS = []
if found:
    print(f"\n{len(found)} rules kept (precision >= {RULE_MIN_PREC}, recall >= {RULE_MIN_REC}):")
    print(pd.DataFrame(found, columns=["feature", "side", "threshold", "precision", "recall", "n"])
          .round(4).to_string(index=False))
    hits = np.zeros(len(df), dtype=np.float32)
    bestp = np.zeros(len(df), dtype=np.float32)
    for i, (f, side, thr, prec, rec, n) in enumerate(found):
        v = df[f].to_numpy(dtype=np.float64)
        ind = (((v >= thr) if side == "hi" else (v <= thr)) & np.isfinite(v)).astype(np.float32)
        col = f"rule_{i:02d}"
        df[col] = ind.astype(np.int8)
        RULE_COLS.append(col)
        hits += ind
        bestp = np.maximum(bestp, ind * prec)
    df["rule_hits"] = hits
    df["rule_best_prec"] = bestp
    RULE_COLS += ["rule_hits", "rule_best_prec"]
    cov = float((hits[IS_TRAIN] > 0).mean())
    caught = float(y_all[IS_TRAIN & (hits > 0)].sum() / y_all[IS_TRAIN].sum())
    print(f"\nrule columns added: {len(RULE_COLS)} | fire on {cov:.4%} of labelled rows"
          f" | cover {caught:.2%} of labelled fraud")
else:
    print("\nno rule cleared the thresholds - no rule columns added")

X_all = df[FEATURES + RULE_COLS] if RULE_COLS else X_all
gc.collect()
tick("rule mining done")


mining rules on 380,245 rows up to 2026-05-13 (base rate 0.0181)

12 rules kept (precision >= 0.25, recall >= 0.005):
           feature side   threshold  precision  recall    n
  dev_outside_pool   hi      1.0000     1.0000  0.2539 1747
amt_over_cust_mean   hi     55.1676     1.0000  0.0270  186
    device_ordinal   hi 918586.6340     1.0000  0.0278  191
    merch_surge_1h   hi      6.4972     1.0000  0.0282  194
   merch_surge_24h   hi     10.0000     0.9899  0.0286  199
cust_rate_ratio_1h   hi     10.9565     0.9895  0.0549  382
    novelty_x_amtz   hi     20.0057     0.9895  0.0275  191
 amt_over_cust_max   hi     35.4836     0.9892  0.0267  186
    cust_gap_ratio   lo      0.0000     0.9677  0.0262  186
   amt_z_cust_mcat   hi      9.2905     0.9580  0.0166  119
  cust_loc_recency   lo      1.9459     0.9529  0.0264  191
 account_age_drift   hi     24.1553     0.9424  0.0262  191

rule columns added: 14 | fire on 1.5788% of labelled rows | cover 39.86% of labelled fraud
[  180.2s]

In [21]:
# ============================================================================
# 19b | How much does a stale encoding cost?
# ============================================================================
# Every fraud-rate encoding in this pipeline is computed from labels up to each row's own
# timestamp. At scoring time they stop on 15 July and go up to 62 days stale. No fold has ever
# been scored under that condition, so the cost is unknown - and it is the difference between
# the folds predicting the leaderboard and flattering it.
#
# `build_te_frozen` rebuilds every encoding with the labelled history cut at a chosen moment.
# Rows before the cut accumulate exactly as before; rows after it inherit the value frozen
# there, and the decayed view additionally relaxes toward the base rate the further past the
# cut a row sits - which is the correct behaviour when the evidence has stopped.
#
# Point the cut at each fold's own fitting boundary and the validation block is placed in
# precisely the position a test row occupies.
TE_SPEC = {"loc": (loc_c, TE_SMOOTH), "merchant": (merc_c, TE_SMOOTH),
           "loc_hour": (lh_key, TE_SMOOTH), "mcat_hour": (composite(mcat_c, hb_c), TE_SMOOTH),
           "customer": (cust_c, TE_SMOOTH_HI), "device": (dev_c, TE_SMOOTH_HI)}
TE_COLS = [f"{p}{n}" for n in TE_SPEC for p in ("te_", "ted_", "ten_")]
TE_COLS = [c for c in TE_COLS if c in FEATURES]
prior_rate = float(y_all[IS_TRAIN].mean())
w_up = np.exp(t_days / TE_TAU)
w_dn = np.exp(-t_days / TE_TAU)
y_num = np.nan_to_num(y_all.to_numpy(dtype=np.float64))

def build_te_frozen(cutoff_ts):
    """Every encoding as it stood at `cutoff_ts`; later rows inherit the frozen value."""
    lab = (IS_TRAIN & (ts_sec <= cutoff_ts)).astype(np.float64)
    cy = y_num * lab
    out = {}
    for nm, (codes, smooth) in TE_SPEC.items():
        n_before = pd.Series(lab).groupby(codes).cumsum().to_numpy() - lab
        p_before = pd.Series(cy).groupby(codes).cumsum().to_numpy() - cy
        if f"te_{nm}" in FEATURES:
            out[f"te_{nm}"] = ((p_before + prior_rate * smooth)
                               / (n_before + smooth)).astype(np.float32)
        if f"ten_{nm}" in FEATURES:
            out[f"ten_{nm}"] = np.log1p(n_before).astype(np.float32)
        if f"ted_{nm}" in FEATURES:
            A = pd.Series(cy * w_up).groupby(codes).cumsum().to_numpy() - cy * w_up
            B = pd.Series(lab * w_up).groupby(codes).cumsum().to_numpy() - lab * w_up
            out[f"ted_{nm}"] = ((A * w_dn + prior_rate * smooth)
                                / (B * w_dn + smooth)).astype(np.float32)
    return out

USE_FEATURES = list(KEPT) + RULE_COLS
USE_CATS = [c for c in CAT_COLS if c in USE_FEATURES]
LABEL_PREFIX = ("te_", "ted_", "ten_")
NOLABEL_FEATURES = [f for f in USE_FEATURES if not f.startswith(LABEL_PREFIX)]
NOLABEL_CATS = [c for c in CAT_COLS if c in NOLABEL_FEATURES]
print(f"full set {len(USE_FEATURES)} | label-free set {len(NOLABEL_FEATURES)} "
      f"({len(USE_FEATURES) - len(NOLABEL_FEATURES)} encodings)")

def lgb_run(frame, tr_mask, va_mask, feats, cats, rounds=None, seed=None):
    p = dict(LGB_PARAMS)
    if seed is not None:
        p.update(seed=seed, bagging_seed=seed, feature_fraction_seed=seed, data_random_seed=seed)
    dtr = lgb.Dataset(frame.loc[tr_mask, feats], label=y_all[tr_mask], categorical_feature=cats)
    if rounds is None:
        dva = lgb.Dataset(frame.loc[va_mask, feats], label=y_all[va_mask],
                          categorical_feature=cats, reference=dtr)
        m = lgb.train(p, dtr, num_boost_round=MAX_ROUNDS, valid_sets=[dva],
                      callbacks=[lgb.early_stopping(EARLY_STOP, verbose=False), lgb.log_evaluation(0)])
        out, it = m.predict(frame.loc[va_mask, feats], num_iteration=m.best_iteration), m.best_iteration
    else:
        m = lgb.train(p, dtr, num_boost_round=int(rounds))
        out, it = m.predict(frame.loc[va_mask, feats]), int(rounds)
    del dtr, m; gc.collect()
    return out, it

print("\nstaleness probe - identical model, identical rows, encodings built two ways:")
fresh_ap, frozen_ap = [], []
for ix, (tr_m, va_m) in enumerate(fold_masks):
    p_fresh, _ = lgb_run(X_all, tr_m, va_m, USE_FEATURES, USE_CATS)
    a_fresh = average_precision_score(y_all[va_m], p_fresh)
    Xf = X_all.assign(**build_te_frozen(FOLD_CUTOFFS[ix]))
    p_froz, _ = lgb_run(Xf, tr_m, va_m, USE_FEATURES, USE_CATS)
    a_froz = average_precision_score(y_all[va_m], p_froz)
    fresh_ap.append(a_fresh); frozen_ap.append(a_froz)
    print(f"  {FOLD_NAMES[ix]:>8}  fresh {a_fresh:.5f}   frozen {a_froz:.5f}"
          f"   cost {a_froz - a_fresh:+.5f}")
    del Xf, p_fresh, p_froz; gc.collect()

STALENESS = selection_score(frozen_ap) - selection_score(fresh_ap)
print(f"\nstaleness cost on the selection score: {STALENESS:+.5f}")
print("  -> the folds materially overstate what the scoring window will give"
      if STALENESS < -0.005 else
      "  -> stale encodings cost little; the folds are a fair guide")
tick("staleness probe done")


full set 264 | label-free set 246 (18 encodings)

staleness probe - identical model, identical rows, encodings built two ways:
   horizon  fresh 0.75283   frozen 0.73562   cost -0.01721
    recent  fresh 0.61492   frozen 0.59606   cost -0.01886

staleness cost on the selection score: -0.01829
  -> the folds materially overstate what the scoring window will give
[ 1433.3s] staleness probe done


In [22]:
# ============================================================================
# 20 | Four members
# ============================================================================
# `nol` carries no fraud-rate encoding at all. It scores lower while encodings are fresh,
# because they carry real gain - and it is immune to whatever the probe above just measured.
# Its mistakes differ from the others' for the same reason, which is what a blend can use.
USE_CAT = USE_MLP = True
try:
    from catboost import CatBoostClassifier, Pool
except ImportError:
    USE_CAT = False; print("catboost unavailable")
try:
    import torch
    import torch.nn as nn
except ImportError:
    USE_MLP = False; print("torch unavailable")

CAT_KW = dict(learning_rate=0.12, depth=6, l2_leaf_reg=6.0, loss_function="Logloss",
              eval_metric="PRAUC", verbose=False, allow_writing_files=False)

members, scores, ROUNDS = {}, {}, {}
truth = [y_all[va].to_numpy() for _, va in fold_masks]

lgb_preds, lgb_iters = [], []
for ix, (tr_m, va_m) in enumerate(fold_masks):
    p, it = lgb_run(X_all, tr_m, va_m, USE_FEATURES, USE_CATS)
    lgb_preds.append(to_rank(p)); lgb_iters.append(it)
    print(f"  lgb {FOLD_NAMES[ix]:>8}  PR-AUC {average_precision_score(truth[ix], p):.5f}   best_iter {it}")
members["lgb"] = lgb_preds
scores["lgb"] = [average_precision_score(truth[f], lgb_preds[f]) for f in range(len(fold_masks))]
ROUNDS["lgb"] = max(200, int(np.mean(lgb_iters) * 1.10))
print(f"LightGBM selection score: {selection_score(scores['lgb']):.5f}")
tick("lightgbm done")

try:
    preds, iters = [], []
    for ix, (tr_m, va_m) in enumerate(fold_masks):
        p, it = lgb_run(X_all, tr_m, va_m, NOLABEL_FEATURES, NOLABEL_CATS)
        preds.append(to_rank(p)); iters.append(it)
        print(f"  nol {FOLD_NAMES[ix]:>8}  PR-AUC {average_precision_score(truth[ix], p):.5f}   best_iter {it}")
    members["nol"] = preds
    scores["nol"] = [average_precision_score(truth[f], preds[f]) for f in range(len(fold_masks))]
    ROUNDS["nol"] = max(200, int(np.mean(iters) * 1.10))
    print(f"Label-free selection score: {selection_score(scores['nol']):.5f}")
except Exception as e:
    print(f"  nol failed ({type(e).__name__}: {e}) - dropped")
tick("label-free lightgbm done")

if USE_CAT:
    try:
        cat_idx = [USE_FEATURES.index(c) for c in USE_CATS]
        X_cat = X_all[USE_FEATURES].assign(**{c: X_all[c].astype(str) for c in USE_CATS})
        preds, iters = [], []
        for ix, (tr_m, va_m) in enumerate(fold_masks):
            m = CatBoostClassifier(iterations=MAX_ROUNDS, random_seed=SEEDS[0], od_type="Iter",
                                   od_wait=EARLY_STOP, **CAT_KW)
            m.fit(Pool(X_cat[tr_m], y_all[tr_m], cat_features=cat_idx),
                  eval_set=Pool(X_cat[va_m], y_all[va_m], cat_features=cat_idx), use_best_model=True)
            p = m.predict_proba(Pool(X_cat[va_m], cat_features=cat_idx))[:, 1]
            preds.append(to_rank(p)); iters.append(m.get_best_iteration() + 1)
            print(f"  cat {FOLD_NAMES[ix]:>8}  PR-AUC {average_precision_score(truth[ix], p):.5f}"
                  f"   best_iter {m.get_best_iteration() + 1}")
            del m; gc.collect()
        members["cat"] = preds
        scores["cat"] = [average_precision_score(truth[f], preds[f]) for f in range(len(fold_masks))]
        ROUNDS["cat"] = max(200, int(np.mean(iters) * 1.10))
        print(f"CatBoost selection score: {selection_score(scores['cat']):.5f}")
    except Exception as e:
        print(f"  cat failed ({type(e).__name__}: {e}) - dropped")
    tick("catboost done")

if USE_MLP:
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"torch device: {DEVICE}")
    MLP_NUM = [c for c in USE_FEATURES if c not in USE_CATS]
    MLP_CARDS = [int(df[c].cat.categories.size) for c in USE_CATS]

    class TabMLP(nn.Module):
        def __init__(self, n_num, cards, widths=(384, 192, 96), p_drop=0.25):
            super().__init__()
            self.emb = nn.ModuleList([nn.Embedding(c, min(8, (c + 1) // 2 + 1)) for c in cards])
            d = n_num + sum(e.embedding_dim for e in self.emb)
            layers = []
            for w in widths:
                layers += [nn.Linear(d, w), nn.BatchNorm1d(w), nn.ReLU(), nn.Dropout(p_drop)]
                d = w
            layers += [nn.Linear(d, 1)]
            self.net = nn.Sequential(*layers)

        def forward(self, xn, xc):
            e = [emb(xc[:, i]) for i, emb in enumerate(self.emb)]
            return self.net(torch.cat([xn] + e, dim=1)).squeeze(-1)

    def mlp_prepare(tr_mask, va_mask):
        A = X_all.loc[tr_mask, MLP_NUM].to_numpy(np.float32)
        B = X_all.loc[va_mask, MLP_NUM].to_numpy(np.float32)
        med = np.nan_to_num(np.nanmedian(A, axis=0))
        iqr = np.nan_to_num(np.nanpercentile(A, 75, axis=0) - np.nanpercentile(A, 25, axis=0),
                            nan=1.0) + 1e-3
        f = lambda Z: np.clip(np.nan_to_num((Z - med) / iqr, nan=0., posinf=0., neginf=0.),
                              -6, 6).astype(np.float32)
        ca = np.clip(np.stack([X_all.loc[tr_mask, c].cat.codes.to_numpy() for c in USE_CATS], 1)
                     .astype(np.int64), 0, None)
        cb = np.clip(np.stack([X_all.loc[va_mask, c].cat.codes.to_numpy() for c in USE_CATS], 1)
                     .astype(np.int64), 0, None)
        out = (f(A), ca, f(B), cb)
        del A, B; gc.collect()
        return out

    @torch.no_grad()
    def mlp_predict(model, vA, vC, bs=32768):
        model.eval()
        out = np.empty(len(vA), dtype=np.float64)
        for s in range(0, len(vA), bs):
            out[s:s + bs] = torch.sigmoid(model(vA[s:s + bs], vC[s:s + bs])).float().cpu().numpy()
        return out

    def run_mlp(prep, ya, yb, seeds, epochs=None):
        A, ca, B, cb = prep
        tA = torch.from_numpy(A).to(DEVICE); tC = torch.from_numpy(ca).to(DEVICE)
        tY = torch.from_numpy(np.asarray(ya, dtype=np.float32)).to(DEVICE)
        vA = torch.from_numpy(B).to(DEVICE); vC = torch.from_numpy(cb).to(DEVICE)
        nb = max(1, int(np.ceil(len(A) / 4096)))
        lossf = nn.BCEWithLogitsLoss()
        acc = np.zeros(len(B)); best_eps = []
        for sd in seeds:
            torch.manual_seed(sd); np.random.seed(sd)
            model = TabMLP(A.shape[1], MLP_CARDS).to(DEVICE)
            opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
            n_ep = int(epochs) if epochs is not None else MLP_MAX_EPOCHS
            sch = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=2e-3, total_steps=n_ep * nb)
            best_ap, best_ep, best_state, stale = -1.0, n_ep, None, 0
            for ep in range(1, n_ep + 1):
                model.train()
                for s in np.array_split(np.random.permutation(len(A)), nb):
                    if len(s) < 2:
                        continue
                    si = torch.from_numpy(s).to(DEVICE)
                    opt.zero_grad()
                    lossf(model(tA[si], tC[si]), tY[si]).backward()
                    opt.step(); sch.step()
                if epochs is not None:
                    continue
                ap = average_precision_score(yb, mlp_predict(model, vA, vC))
                if ap > best_ap:
                    best_ap, best_ep, stale = ap, ep, 0
                    best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
                else:
                    stale += 1
                    if stale >= MLP_PATIENCE:
                        break
            if best_state is not None:
                model.load_state_dict(best_state)
            acc += to_rank(mlp_predict(model, vA, vC)) / len(seeds)
            best_eps.append(best_ep)
            del model, opt
            if DEVICE.type == "cuda":
                torch.cuda.empty_cache()
            gc.collect()
        del tA, tC, tY, vA, vC; gc.collect()
        return acc, int(max(3, round(float(np.mean(best_eps)) * 1.1)))

    try:
        preds, eps = [], []
        for ix, (tr_m, va_m) in enumerate(fold_masks):
            prep = mlp_prepare(tr_m, va_m)
            p, ep = run_mlp(prep, y_all[tr_m].to_numpy(), y_all[va_m].to_numpy(), MLP_VAL_SEEDS)
            preds.append(p); eps.append(ep)
            print(f"  mlp {FOLD_NAMES[ix]:>8}  PR-AUC {average_precision_score(truth[ix], p):.5f}"
                  f"   best_epochs {ep}")
            del prep; gc.collect()
        members["mlp"] = preds
        scores["mlp"] = [average_precision_score(truth[f], preds[f]) for f in range(len(fold_masks))]
        ROUNDS["mlp"] = int(np.median(eps))
        print(f"Network selection score: {selection_score(scores['mlp']):.5f}")
    except Exception as e:
        print(f"  mlp failed ({type(e).__name__}: {e}) - dropped")
    tick("network done")

print(f"\nmembers available: {list(members)}")
tick("members done")


  lgb  horizon  PR-AUC 0.75283   best_iter 180
  lgb   recent  PR-AUC 0.61492   best_iter 239
LightGBM selection score: 0.66319
[ 1760.8s] lightgbm done
  nol  horizon  PR-AUC 0.74849   best_iter 191
  nol   recent  PR-AUC 0.61281   best_iter 329
Label-free selection score: 0.66030
[ 2091.1s] label-free lightgbm done
  cat  horizon  PR-AUC 0.75347   best_iter 357
  cat   recent  PR-AUC 0.61507   best_iter 297
CatBoost selection score: 0.66351
[ 2639.2s] catboost done
torch device: cpu
  mlp  horizon  PR-AUC 0.75155   best_epochs 14
  mlp   recent  PR-AUC 0.61480   best_epochs 11
Network selection score: 0.66266
[ 3010.8s] network done

members available: ['lgb', 'nol', 'cat', 'mlp']
[ 3010.8s] members done


In [23]:
# ============================================================================
# 21 | Blend weights by coordinate ascent
# ============================================================================
# The exhaustive grid cost 795s per run and equal weights landed within 0.0001 of its answer
# every single time. Coordinate ascent reaches the same place in a few hundred evaluations.
names = list(members)
val_ranks = {n: [to_rank(p) for p in members[n]] for n in names}

def blend_aps(w):
    out = []
    for f in range(len(fold_masks)):
        b = np.zeros(len(truth[f]))
        for wi, n in zip(w, names):
            b += wi * val_ranks[n][f]
        out.append(average_precision_score(truth[f], b))
    return out

print("per-model scores (fold order:", ", ".join(FOLD_NAMES) + ")")
single = {}
for n in names:
    single[n] = selection_score(scores[n])
    print(f"  {n:>4}: " + " ".join(f"{a:.4f}" for a in scores[n]) + f"   selection {single[n]:.5f}")
BEST_SINGLE = max(single, key=single.get)

GRID = np.arange(0.0, 1.0001, 0.05)
w = np.ones(len(names)) / len(names)
best_sel = selection_score(blend_aps(w))
for _ in range(4):
    improved = False
    for j in range(len(names)):
        for g in GRID:
            cand = w.copy(); cand[j] = g
            if cand.sum() <= 0:
                continue
            cand = cand / cand.sum()
            s = selection_score(blend_aps(cand))
            if s > best_sel + 1e-9:
                best_sel, w, improved = s, cand, True
    if not improved:
        break
best_w = w
best_aps = blend_aps(best_w)
eq = np.ones(len(names)) / len(names)
sel_eq = selection_score(blend_aps(eq))
print(f"\nweights: {dict(zip(names, np.round(best_w, 3)))}")
print("blended per fold: " + " ".join(f"{a:.4f}" for a in best_aps))
print(f"blend selection {best_sel:.5f} | equal {sel_eq:.5f} | best single {BEST_SINGLE} {single[BEST_SINGLE]:.5f}")
print(f"\nstaleness cost measured earlier: {STALENESS:+.5f}")
print(f"label-free member weight: {best_w[names.index('nol')]:.3f}" if "nol" in names else "")
tick("blend selected")


per-model scores (fold order: horizon, recent)
   lgb: 0.7528 0.6149   selection 0.66319
   nol: 0.7485 0.6128   selection 0.66030
   cat: 0.7535 0.6151   selection 0.66351
   mlp: 0.7516 0.6148   selection 0.66266

weights: {'lgb': np.float64(0.051), 'nol': np.float64(0.195), 'cat': np.float64(0.339), 'mlp': np.float64(0.415)}
blended per fold: 0.7568 0.6181
blend selection 0.66664 | equal 0.66631 | best single cat 0.66351

staleness cost measured earlier: -0.01829
label-free member weight: 0.195
[ 3028.2s] blend selected


In [24]:
# ============================================================================
# 22 | Refit on the full history, score the test window, and write the entries
# ============================================================================
X_test_idx = ~IS_TRAIN
n_test = int(X_test_idx.sum())
y_fit = y_all[FIT_OK]
test_parts = {}

for name, feats, cats in [("lgb", USE_FEATURES, USE_CATS),
                          ("nol", NOLABEL_FEATURES, NOLABEL_CATS)]:
    if name not in members:
        continue
    acc = np.zeros(n_test)
    for sd in SEEDS:
        p, _ = lgb_run(X_all, FIT_OK, X_test_idx, feats, cats, rounds=ROUNDS[name], seed=sd)
        acc += to_rank(p) / len(SEEDS)
    test_parts[name] = acc
    print(f"  {name} fitted over {len(SEEDS)} seeds ({ROUNDS[name]} rounds, {len(feats)} features)")

if "cat" in members:
    fit_pool  = Pool(X_cat[FIT_OK], y_fit, cat_features=cat_idx)
    test_pool = Pool(X_cat[X_test_idx], cat_features=cat_idx)
    acc = np.zeros(n_test)
    for sd in SEEDS:
        m = CatBoostClassifier(iterations=ROUNDS["cat"], random_seed=sd,
                               **{k: v for k, v in CAT_KW.items() if k != "eval_metric"})
        m.fit(fit_pool)
        acc += to_rank(m.predict_proba(test_pool)[:, 1]) / len(SEEDS)
        print(f"  cat seed {sd} fitted ({ROUNDS['cat']} rounds)")
        del m; gc.collect()
    test_parts["cat"] = acc
    del fit_pool, test_pool; gc.collect()

if "mlp" in members:
    prep = mlp_prepare(FIT_OK, X_test_idx)
    p, _ = run_mlp(prep, y_fit.to_numpy(), np.zeros(n_test), MLP_FIT_SEEDS, epochs=ROUNDS["mlp"])
    test_parts["mlp"] = p
    print(f"  mlp fitted over {len(MLP_FIT_SEEDS)} seeds ({ROUNDS['mlp']} epochs)")
    del prep; gc.collect()

test_ids = df.loc[X_test_idx, "transaction_id"].to_numpy()

def finalise(scores_arr, path):
    """Rank-normalise into the open unit interval; PR-AUC scores the ordering only."""
    r = (rankdata(scores_arr) - 0.5) / len(scores_arr)
    out = pd.DataFrame({"transaction_id": test_ids, "fraud": r})
    out = sample_sub[["transaction_id"]].merge(out, on="transaction_id", how="left")
    out["fraud"] = out["fraud"].fillna(out["fraud"].median()).clip(0.0, 1.0)
    out.to_csv(path, index=False)
    return out

blend_test = np.zeros(n_test)
for wq, n in zip(best_w, names):
    blend_test += wq * test_parts[n]
sub_blend = finalise(blend_test, "submission_blend_v3v9.csv")

# Second entry. If the probe says stale encodings cost real ground, the hedge is the member
# that owns none of them; otherwise it is the untuned average, which matched the tuned weights
# to four decimals in every previous run.
if "nol" in test_parts and STALENESS < -0.005:
    alt, alt_label, alt_path = test_parts["nol"], "label-free model", "submission_nolabel_v3v9.csv"
else:
    alt = np.zeros(n_test)
    for n in names:
        alt += test_parts[n] / len(names)
    alt_label, alt_path = "equal-weight blend", "submission_equal_v3v9.csv"
finalise(alt, alt_path)
sub_blend.to_csv("submission.csv", index=False)

print("\n" + sub_blend.head().to_string())
print("\nrows:", len(sub_blend), "| expected:", n_test,
      "| nulls:", int(sub_blend["fraud"].isna().sum()))
print(f"\nblend -> horizon {best_aps[HORIZON_IX]:.5f}  recent {best_aps[RECENT_IX]:.5f}"
      f"  selection {best_sel:.5f}")
print(f"staleness cost {STALENESS:+.5f} | entry 2 = {alt_label} ({alt_path})")
print(f"members {names} | features {len(USE_FEATURES)} | label-free {len(NOLABEL_FEATURES)}")
print("reference on the same blocks:")
print("  abdur v3  horizon 0.75051   (w4 0.61970)")
print("  fraud11   horizon 0.75624   recent 0.61661")
print("  v3_v5     horizon 0.75189   recent 0.61591")
tick("submissions written")


  lgb fitted over 2 seeds (230 rounds, 264 features)
  nol fitted over 2 seeds (286 rounds, 246 features)
  cat seed 42 fitted (359 rounds)
  cat seed 202 fitted (359 rounds)
  mlp fitted over 2 seeds (12 epochs)

  transaction_id     fraud
0     T000731942  0.208951
1     T000731943  0.279410
2     T000731944  0.031519
3     T000731945  0.030788
4     T000731946  0.151855

rows: 262648 | expected: 262648 | nulls: 0

blend -> horizon 0.75682  recent 0.61808  selection 0.66664
staleness cost -0.01829 | entry 2 = label-free model (submission_nolabel_v3v9.csv)
members ['lgb', 'nol', 'cat', 'mlp'] | features 264 | label-free 246
reference on the same blocks:
  abdur v3  horizon 0.75051   (w4 0.61970)
  fraud11   horizon 0.75624   recent 0.61661
  v3_v5     horizon 0.75189   recent 0.61591
[ 4377.2s] submissions written
